# UH Mānoa Substation — Encoder-Decoder LSTM Load Forecast
**24-hour horizon · 15-min data resampled to 1-hour · FY 2025**

| Hyperparameter | Value |
|---|---|
| Encoder look-back | 24 h |
| Decoder horizon | 24 h |
| Hidden units | 128 |
| Layers | 2 LSTM + LayerNorm |
| Loss | Weighted MSE (daytime 2.5×) |
| Features | 13 (weather + cyclical + lags + session flag) |

## 1. Setup

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import requests
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42); np.random.seed(42)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}  |  torch {torch.__version__}')

DATA_PATH    = os.path.join(os.path.expanduser('~'), 'Downloads',
               'substation_totalized_kw_15_min_fy25.csv')
SEQ_LEN      = 24
PRED_HORIZON = 24
HIDDEN       = 128
NUM_LAYERS   = 2
DROPOUT      = 0.25
BATCH_SIZE   = 64
EPOCHS       = 80
LR           = 1e-3
TRAIN_RATIO  = 0.70
VAL_RATIO    = 0.15
DAY_W        = 2.5


## 2. Load & Resample
Resample 15-min intervals to hourly means.

In [ ]:
df_raw = pd.read_csv(DATA_PATH, parse_dates=['datetime'])
df_raw = df_raw.set_index('datetime').sort_index()

df = df_raw[['kw']].resample('1h').mean()
df.index = df.index.tz_localize(None)

print(f'Hourly samples : {len(df)}')
print(f'Date range     : {df.index[0]}  →  {df.index[-1]}')
print(f'kW  min={df["kw"].min():.1f}  max={df["kw"].max():.1f}  nulls={df["kw"].isna().sum()}')

df['kw'].plot(figsize=(14, 3), title='UH Mānoa Substation – Hourly kW (FY 2025)')
plt.ylabel('kW'); plt.tight_layout(); plt.show()


## 3. Weather Data
Pull temperature, solar irradiance (W/m²), and relative humidity from the Open-Meteo historical archive for Honolulu.

In [ ]:
params = {
    'latitude': 21.3069, 'longitude': -157.8583,
    'start_date': '2024-07-01', 'end_date': '2025-06-30',
    'hourly': 'temperature_2m,shortwave_radiation,relative_humidity_2m',
    'timezone': 'Pacific/Honolulu',
}
resp = requests.get(
    'https://archive-api.open-meteo.com/v1/archive', params=params, timeout=60
)
resp.raise_for_status()
wx = resp.json()['hourly']

df_wx = pd.DataFrame({
    'datetime' : pd.to_datetime(wx['time']),
    'temp_c'   : wx['temperature_2m'],
    'solar_wm2': wx['shortwave_radiation'],
    'rh'       : wx['relative_humidity_2m'],
})
df_wx = df_wx.set_index('datetime')
df_wx.index = df_wx.index.tz_localize(None)
print(f'Weather rows: {len(df_wx)}')
df_wx.head()


## 4. Feature Engineering
- **Cyclical encoding** — hour, day-of-week, month via sin/cos so the model sees periodicity without a discontinuity at midnight/Sunday/January.
- **`is_school_session`** — binary flag for Fall 2024 and Spring 2025 semesters; campus load drops significantly during breaks.
- **Autoregressive lags** — `kw_lag24` (same hour yesterday) and `kw_lag168` (same hour last week) give the model direct access to the most predictive prior observations.

In [ ]:
df = df.join(df_wx, how='inner')
print(f'After join: {len(df)} rows')

hour  = df.index.hour
dow   = df.index.dayofweek
month = df.index.month

df['hour_sin']  = np.sin(2 * np.pi * hour  / 24)
df['hour_cos']  = np.cos(2 * np.pi * hour  / 24)
df['dow_sin']   = np.sin(2 * np.pi * dow   / 7)
df['dow_cos']   = np.cos(2 * np.pi * dow   / 7)
df['month_sin'] = np.sin(2 * np.pi * (month - 1) / 12)
df['month_cos'] = np.cos(2 * np.pi * (month - 1) / 12)

session_ranges = [
    ('2024-08-26', '2024-12-14'),
    ('2025-01-13', '2025-05-10'),
]
df['is_school_session'] = 0
for start, end in session_ranges:
    df.loc[(df.index >= start) & (df.index <= end), 'is_school_session'] = 1
print(f'School-session hours: {df["is_school_session"].sum()} / {len(df)}')

df['kw_lag24']  = df['kw'].shift(24)
df['kw_lag168'] = df['kw'].shift(168)
df.dropna(inplace=True)
print(f'Usable rows after lag padding: {len(df)}')

FEATURE_COLS = [
    'kw', 'temp_c', 'solar_wm2', 'rh',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'month_sin', 'month_cos',
    'is_school_session', 'kw_lag24', 'kw_lag168',
]
TARGET_COL = 'kw'
TARGET_IDX = FEATURE_COLS.index(TARGET_COL)
print(f'Features ({len(FEATURE_COLS)}): {FEATURE_COLS}')


## 5. Scale & Dataset
`RobustScaler` uses median + IQR, so future negative-kW solar-export readings won't distort the scaling of the nighttime baseline.

In [ ]:
n       = len(df)
n_train = int(n * TRAIN_RATIO)
n_val   = int(n * VAL_RATIO)
n_test  = n - n_train - n_val

df_train = df.iloc[:n_train]
df_val   = df.iloc[n_train : n_train + n_val]
df_test  = df.iloc[n_train + n_val:]
print(f'Train: {len(df_train)}  Val: {len(df_val)}  Test: {len(df_test)}')

scaler = RobustScaler()
scaler.fit(df_train[FEATURE_COLS].values)

arr_train = scaler.transform(df_train[FEATURE_COLS].values)
arr_val   = scaler.transform(df_val  [FEATURE_COLS].values)
arr_test  = scaler.transform(df_test [FEATURE_COLS].values)


class SeqDataset(Dataset):
    def __init__(self, arr, seq_len=SEQ_LEN, horizon=PRED_HORIZON):
        self.arr     = torch.tensor(arr, dtype=torch.float32)
        self.seq_len = seq_len
        self.horizon = horizon

    def __len__(self):
        return len(self.arr) - self.seq_len - self.horizon + 1

    def __getitem__(self, idx):
        x = self.arr[idx : idx + self.seq_len]
        y = self.arr[idx + self.seq_len : idx + self.seq_len + self.horizon, TARGET_IDX]
        return x, y


ds_train = SeqDataset(arr_train)
ds_val   = SeqDataset(arr_val)
ds_test  = SeqDataset(arr_test)

dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
dl_val   = DataLoader(ds_val,   batch_size=BATCH_SIZE, shuffle=False)
dl_test  = DataLoader(ds_test,  batch_size=BATCH_SIZE, shuffle=False)
print(f'Batches — train: {len(dl_train)}  val: {len(dl_val)}  test: {len(dl_test)}')


## 6. Model — Encoder-Decoder LSTM
**Why LayerNorm instead of BatchNorm?**  
BatchNorm normalises across the batch dimension. For LSTM hidden states this is unstable with small batches and breaks at inference when batch size = 1. LayerNorm normalises per sample across the hidden dimension — no batch dependency.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_size, hidden, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden, num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.ln = nn.LayerNorm(hidden)

    def forward(self, x):
        _, (h, c) = self.lstm(x)
        h = self.ln(h)
        return h, c


class Decoder(nn.Module):
    def __init__(self, input_size, hidden, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden, num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.ln = nn.LayerNorm(hidden)
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x, h, c):
        out, (h, c) = self.lstm(x, (h, c))
        out = self.ln(out)
        return self.fc(out).squeeze(-1), h, c


class EncoderDecoderLSTM(nn.Module):
    def __init__(
        self,
        num_features=len(FEATURE_COLS),
        hidden=HIDDEN,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
        pred_horizon=PRED_HORIZON,
    ):
        super().__init__()
        self.pred_horizon = pred_horizon
        self.encoder = Encoder(num_features, hidden, num_layers, dropout)
        self.decoder = Decoder(1, hidden, num_layers, dropout)

    def forward(self, src, teacher_target=None, teacher_forcing_ratio=0.5):
        h, c = self.encoder(src)
        dec_in = src[:, -1, TARGET_IDX].unsqueeze(1).unsqueeze(2)
        outputs = []
        for t in range(self.pred_horizon):
            pred_t, h, c = self.decoder(dec_in, h, c)
            outputs.append(pred_t)
            use_teacher = (
                teacher_target is not None
                and torch.rand(1).item() < teacher_forcing_ratio
            )
            dec_in = (
                teacher_target[:, t].unsqueeze(1).unsqueeze(2)
                if use_teacher else pred_t.unsqueeze(2)
            )
        return torch.cat(outputs, dim=1)


model = EncoderDecoderLSTM().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}')
print(model)


## 7. Train
**Weighted MSE** gives hours 07–18 HST a 2.5× penalty so the model pays extra attention to the solar-ramp period where errors matter most for grid dispatch.

In [ ]:
_w = torch.ones(PRED_HORIZON, device=DEVICE)
for _h in range(PRED_HORIZON):
    if 7 <= _h <= 18:
        _w[_h] = DAY_W

def weighted_mse(pred, target, weights=_w):
    return (((pred - target) ** 2) * weights).mean()


optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val  = float('inf')
ckpt_path = os.path.join(os.path.dirname(DATA_PATH), 'best_lstm.pth')
history   = {'train': [], 'val': []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    t_loss = 0.0
    for X, y in dl_train:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = weighted_mse(model(X, teacher_target=y), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        t_loss += loss.item() * X.size(0)
    t_loss /= len(ds_train)

    model.eval()
    v_loss = 0.0
    with torch.no_grad():
        for X, y in dl_val:
            X, y = X.to(DEVICE), y.to(DEVICE)
            v_loss += weighted_mse(model(X), y).item() * X.size(0)
    v_loss /= len(ds_val)

    history['train'].append(t_loss)
    history['val'].append(v_loss)
    scheduler.step()

    if v_loss < best_val:
        best_val = v_loss
        torch.save(model.state_dict(), ckpt_path)

    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d}/{EPOCHS}  train={t_loss:.4f}  val={v_loss:.4f}')

model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
print(f'\nBest val loss: {best_val:.4f}')

plt.figure(figsize=(9, 3))
plt.plot(history['train'], label='Train')
plt.plot(history['val'],   label='Val')
plt.xlabel('Epoch'); plt.ylabel('Weighted MSE')
plt.title('Training history'); plt.legend()
plt.tight_layout(); plt.show()


## 8. Evaluate

In [ ]:
def inv_kw(scaled: np.ndarray) -> np.ndarray:
    dummy = np.zeros((len(scaled), len(FEATURE_COLS)))
    dummy[:, TARGET_IDX] = scaled
    return scaler.inverse_transform(dummy)[:, TARGET_IDX]


model.eval()
preds_s, trues_s = [], []
with torch.no_grad():
    for X, y in dl_test:
        preds_s.append(model(X.to(DEVICE)).cpu().numpy())
        trues_s.append(y.numpy())

preds_s = np.concatenate(preds_s)
trues_s = np.concatenate(trues_s)

pred_kw = np.stack([inv_kw(preds_s[:, h]) for h in range(PRED_HORIZON)], axis=1)
true_kw = np.stack([inv_kw(trues_s[:, h]) for h in range(PRED_HORIZON)], axis=1)

rmse = np.sqrt(mean_squared_error(true_kw.ravel(), pred_kw.ravel()))
mae  = mean_absolute_error(true_kw.ravel(), pred_kw.ravel())
mape = np.mean(np.abs((true_kw - pred_kw) / (np.abs(true_kw) + 1e-6))) * 100
print(f'Test  RMSE: {rmse:.1f} kW   MAE: {mae:.1f} kW   MAPE: {mape:.2f}%')

fig, axes = plt.subplots(3, 1, figsize=(14, 11))

axes[0].plot(true_kw[:, 0], label='Actual', alpha=0.8)
axes[0].plot(pred_kw[:, 0], label='Predicted (+1 h)', alpha=0.7)
axes[0].set_title('Test set — 1-hour-ahead forecast')
axes[0].set_ylabel('kW'); axes[0].legend()

h_rmse = [np.sqrt(mean_squared_error(true_kw[:, h], pred_kw[:, h]))
          for h in range(PRED_HORIZON)]
axes[1].bar(range(1, PRED_HORIZON + 1), h_rmse, color='steelblue')
axes[1].set_xlabel('Forecast horizon (h)'); axes[1].set_ylabel('RMSE (kW)')
axes[1].set_title('RMSE degradation across forecast horizon')

axes[2].plot(range(1, 25), true_kw[-1], 'o-', label='Actual')
axes[2].plot(range(1, 25), pred_kw[-1], 's--', label='Predicted')
axes[2].set_xlabel('Hour'); axes[2].set_ylabel('kW')
axes[2].set_title('Example 24-h forecast (last test sample)')
axes[2].legend()

plt.tight_layout(); plt.show()


## 9. Permutation Feature Importance
Shuffle one feature at a time across all test windows and measure how much RMSE increases. A large increase means the model relied heavily on that feature.

In [ ]:
baseline_rmse = rmse


def eval_permuted(feat_idx: int) -> float:
    arr_perm = arr_test.copy()
    arr_perm[:, feat_idx] = arr_perm[
        np.random.permutation(len(arr_perm)), feat_idx
    ]
    ds_p = SeqDataset(arr_perm)
    dl_p = DataLoader(ds_p, batch_size=BATCH_SIZE, shuffle=False)
    ps, ts = [], []
    model.eval()
    with torch.no_grad():
        for X, y in dl_p:
            ps.append(model(X.to(DEVICE)).cpu().numpy())
            ts.append(y.numpy())
    ps = np.concatenate(ps); ts = np.concatenate(ts)
    pk = np.stack([inv_kw(ps[:, h]) for h in range(PRED_HORIZON)], axis=1)
    tk = np.stack([inv_kw(ts[:, h]) for h in range(PRED_HORIZON)], axis=1)
    return np.sqrt(mean_squared_error(tk.ravel(), pk.ravel()))


print('Computing permutation importance (~1-2 min)…')
importances = {}
for i, feat in enumerate(FEATURE_COLS):
    perm_rmse = eval_permuted(i)
    importances[feat] = (perm_rmse - baseline_rmse) / baseline_rmse
    print(f'  {feat:<22s}  delta RMSE%: {importances[feat]*100:+.1f}%')

sorted_feats = sorted(importances, key=importances.get, reverse=True)
sorted_vals  = [importances[f] * 100 for f in sorted_feats]

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['tomato' if v > 0 else 'steelblue' for v in sorted_vals]
ax.barh(sorted_feats[::-1], sorted_vals[::-1], color=colors[::-1])
ax.axvline(0, color='k', linewidth=0.8)
ax.set_xlabel('Relative RMSE change when feature is shuffled (%)')
ax.set_title(
    'Permutation Feature Importance\n'
    '(positive = model degrades without this feature)'
)
plt.tight_layout(); plt.show()
